In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
print(os.getcwd())

c:\Users\User\OneDrive\Desktop\Research\Agentic-AI-Based-Market-Intelligence-Framework-For-Bitcoin-and-Ethereum-Trading\Component_4\notebooks


In [3]:
files = [
    "BTCUSDT_15m.csv",
    "BTCUSDT_1h.csv",
    "BTCUSDT_4h.csv",
    "ETHUSDT_15m.csv",
    "ETHUSDT_1h.csv",
    "ETHUSDT_4h.csv"
]

In [4]:
strong_data = []

for file in files:
    df = pd.read_csv("../data/ml/strong_zone/" + file)
    strong_data.append(df)

strong_df = pd.concat(strong_data, ignore_index=True)

print("Strong Zone:", strong_df.shape)

Strong Zone: (154751, 32)


In [5]:
scenario_data = []

for file in files:
    df = pd.read_csv("../data/ml/scenario/" + file)
    scenario_data.append(df)

scenario_df = pd.concat(scenario_data, ignore_index=True)

print("Scenario:", scenario_df.shape)

Scenario: (274722, 32)


In [6]:
print(strong_df.columns.tolist())
print()
print(scenario_df.columns.tolist())

['sample_id', 'model', 'registry_zone_id', 'zone_version_at_utc', 'source_file', 'source_row', 'symbol', 'source_timeframe', 'zone_family', 'origin_type', 'direction', 'lower_price', 'upper_price', 'decision_at_utc', 'decision_atr', 'touch_at_utc', 'observation_end_utc', 'label', 'label_status', 'split', 'split_status', 'width_atr', 'signed_distance_atr', 'normalized_atr', 'body_ratio', 'close_location_value', 'relative_volume_20', 'rolling_mean_return_20', 'version_age_bars', 'touch_penetration', 'overlapping_family_count_at_decision', 'overlapping_candidate_count_at_decision']

['sample_id', 'model', 'registry_zone_id', 'zone_version_at_utc', 'source_file', 'source_row', 'symbol', 'source_timeframe', 'zone_family', 'origin_type', 'direction', 'lower_price', 'upper_price', 'decision_at_utc', 'decision_atr', 'touch_at_utc', 'observation_end_utc', 'label', 'label_status', 'split', 'split_status', 'width_atr', 'signed_distance_atr', 'normalized_atr', 'body_ratio', 'close_location_value',

Keep only valid labels

In [7]:
strong_df = strong_df[
    (strong_df["label_status"] == "labeled") &
    (strong_df["split_status"] == "eligible")
].copy()

scenario_df = scenario_df[
    (scenario_df["label_status"] == "labeled") &
    (scenario_df["split_status"] == "eligible")
].copy()

print("Strong Zone:", len(strong_df))
print("Scenario:", len(scenario_df))

Strong Zone: 99890
Scenario: 150201


Seperate

In [8]:
strong_train = strong_df[strong_df["split"] == "train"]
strong_val = strong_df[strong_df["split"] == "validation"]
strong_test = strong_df[strong_df["split"] == "test"]

scenario_train = scenario_df[scenario_df["split"] == "train"]
scenario_val = scenario_df[scenario_df["split"] == "validation"]
scenario_test = scenario_df[scenario_df["split"] == "test"]

print(len(strong_train), len(strong_val), len(strong_test))
print(len(scenario_train), len(scenario_val), len(scenario_test))

70525 17571 11794
106588 25968 17645


class distribution

In [9]:
print("Strong Zone:")
print(strong_train["label"].value_counts())

print("\nScenario:")
print(scenario_train["label"].value_counts())

Strong Zone:
label
0.0    49477
1.0    21048
Name: count, dtype: int64

Scenario:
label
bounce           58196
breakout         45912
consolidation     2480
Name: count, dtype: int64


Add weight

In [10]:
from sklearn.utils.class_weight import compute_class_weight

strong_classes = np.unique(strong_train["label"])

strong_weights = compute_class_weight(
    class_weight="balanced",
    classes=strong_classes,
    y=strong_train["label"]
)

print(dict(zip(strong_classes, strong_weights)))

{np.float64(0.0): np.float64(0.7127048931826909), np.float64(1.0): np.float64(1.6753373242113265)}


In [11]:
scenario_classes = np.unique(scenario_train["label"])

scenario_weights = compute_class_weight(
    class_weight="balanced",
    classes=scenario_classes,
    y=scenario_train["label"]
)

print(dict(zip(scenario_classes, scenario_weights)))

{'bounce': np.float64(0.6105116044630788), 'breakout': np.float64(0.7738572341290585), 'consolidation': np.float64(14.326344086021505)}


Model A features 14
Model B features 16

In [12]:
categorical = [
    "symbol",
    "source_timeframe",
    "zone_family",
    "origin_type",
    "direction"
]

numeric = [
    "width_atr",
    "signed_distance_atr",
    "normalized_atr",
    "body_ratio",
    "close_location_value",
    "relative_volume_20",
    "rolling_mean_return_20",
    "overlapping_family_count_at_decision",
    "overlapping_candidate_count_at_decision"
]

strong_features = categorical + numeric

scenario_features = strong_features + [
    "version_age_bars",
    "touch_penetration"
]

print(len(strong_features))
print(len(scenario_features))

14
16


Strong Zone inputs and labels

In [13]:
X_strong_train = strong_train[strong_features]
X_strong_val = strong_val[strong_features]
X_strong_test = strong_test[strong_features]

y_strong_train = strong_train["label"]
y_strong_val = strong_val["label"]
y_strong_test = strong_test["label"]

Scenario inputs and labels

In [14]:
X_scenario_train = scenario_train[scenario_features]
X_scenario_val = scenario_val[scenario_features]
X_scenario_test = scenario_test[scenario_features]

y_scenario_train = scenario_train["label"]
y_scenario_val = scenario_val["label"]
y_scenario_test = scenario_test["label"]

In [15]:
print(X_strong_train.isnull().sum())


symbol                                     0
source_timeframe                           0
zone_family                                0
origin_type                                0
direction                                  0
width_atr                                  0
signed_distance_atr                        0
normalized_atr                             0
body_ratio                                 0
close_location_value                       0
relative_volume_20                         4
rolling_mean_return_20                     5
overlapping_family_count_at_decision       0
overlapping_candidate_count_at_decision    0
dtype: int64


In [16]:
print(X_scenario_train.isnull().sum())

symbol                                     0
source_timeframe                           0
zone_family                                0
origin_type                                0
direction                                  0
width_atr                                  0
signed_distance_atr                        0
normalized_atr                             0
body_ratio                                 0
close_location_value                       0
relative_volume_20                         5
rolling_mean_return_20                     6
overlapping_family_count_at_decision       0
overlapping_candidate_count_at_decision    0
version_age_bars                           0
touch_penetration                          0
dtype: int64


numeric missing values fill using median, categorical values hot encoding

In [17]:
from sklearn.preprocessing import StandardScaler

numeric_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

strong_preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessing, numeric),
    ("categorical", categorical_preprocessing, categorical)
])

In [18]:
scenario_numeric = numeric + [
    "version_age_bars",
    "touch_penetration"
]

scenario_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), scenario_numeric),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]), categorical)
])

In [19]:
strong_train_ready = strong_preprocessor.fit_transform(X_strong_train)

strong_val_ready = strong_preprocessor.transform(X_strong_val)

strong_test_ready = strong_preprocessor.transform(X_strong_test)

print(strong_train_ready.shape)
print(strong_val_ready.shape)
print(strong_test_ready.shape)

(70525, 26)
(17571, 26)
(11794, 26)


In [20]:
scenario_train_ready = scenario_preprocessor.fit_transform(X_scenario_train)

scenario_val_ready = scenario_preprocessor.transform(X_scenario_val)

scenario_test_ready = scenario_preprocessor.transform(X_scenario_test)

print(scenario_train_ready.shape)
print(scenario_val_ready.shape)
print(scenario_test_ready.shape)

(106588, 28)
(25968, 28)
(17645, 28)


In [ ]:
os.makedirs("../data/preprocess/strong_zone", exist_ok=True)
os.makedirs("../data/preprocess/scenario", exist_ok=True)
os.makedirs("../models/preprocess", exist_ok=True)

In [25]:
strong_columns = strong_preprocessor.get_feature_names_out()

scenario_columns = scenario_preprocessor.get_feature_names_out()

In [26]:
strong_train_df = pd.DataFrame(strong_train_ready, columns=strong_columns)
strong_val_df = pd.DataFrame(strong_val_ready, columns=strong_columns)
strong_test_df = pd.DataFrame(strong_test_ready, columns=strong_columns)

strong_train_df["label"] = y_strong_train.to_numpy()
strong_val_df["label"] = y_strong_val.to_numpy()
strong_test_df["label"] = y_strong_test.to_numpy()

In [27]:
scenario_train_df = pd.DataFrame(scenario_train_ready, columns=scenario_columns)
scenario_val_df = pd.DataFrame(scenario_val_ready, columns=scenario_columns)
scenario_test_df = pd.DataFrame(scenario_test_ready, columns=scenario_columns)

scenario_train_df["label"] = y_scenario_train.to_numpy()
scenario_val_df["label"] = y_scenario_val.to_numpy()
scenario_test_df["label"] = y_scenario_test.to_numpy()

In [28]:
strong_train_df.to_csv("../data/preprocess/strong_zone/train.csv", index=False)
strong_val_df.to_csv("../data/preprocess/strong_zone/validation.csv", index=False)
strong_test_df.to_csv("../data/preprocess/strong_zone/test.csv", index=False)

In [29]:
scenario_train_df.to_csv("../data/preprocess/scenario/train.csv", index=False)
scenario_val_df.to_csv("../data/preprocess/scenario/validation.csv", index=False)
scenario_test_df.to_csv("../data/preprocess/scenario/test.csv", index=False)

Fitted preprocessors save

In [30]:
joblib.dump(
    strong_preprocessor,
    "../models/preprocess/strong_preprocessor.joblib"
)

joblib.dump(
    scenario_preprocessor,
    "../models/preprocess/scenario_preprocessor.joblib"
)

['../models/preprocess/scenario_preprocessor.joblib']